# Introduction

Cart abandonment is a critical challenge for digital ordering platforms, directly impacting revenue and customer retention. For MyCoke360, Coca-Cola's B2B digital ordering system launched in Summer 2024, understanding why customers fail to complete purchases is especially important. The platform serves Food Service On Premise (FSOP) customers such as restaurants, schools, hospitals, and retailers, where order frequency and product mix drive significant business value. By examining customer behavior captured in Google Analytics alongside order and sales data, this project seeks to uncover patterns that explain when, how, and why carts are abandoned.

This exploratory data analysis (EDA) will focus on evaluating the quality, structure, and usability of the available data so that it is ready to be used for financial evaluation modeling in the later modeling stage. Other aspects of the problem statement such as identifying behavioral predictors, analyzing recovery patterns, and evaluating device-specific abandonment will be addressed by other members of the project team. This division of workflow ensures comprehensive coverage of the problem space while allowing each stage of the analysis to build on a solid data foundation.

## Initial Guiding Questions

- What is the most efficient method for representing the previously defined cart abondonment within the data?
- How can abandoned carts be aggregated to accurately estimate lost revenue at both the order and product level?
- Which product categories, pack types, or SKUs appear most frequently in abandoned carts, and how should these be visualized for clear insights?
- What is the distribution of abandonment across different customer segments, such as sales office, plant, or FSOP type?
- How can abandoned cart revenue be compared against total sales to highlight the relative financial impact?
- What temporal patterns emerge in abandoned carts (e.g., by day of week, order cycle, or over time during the study period)?
- Are there systematic differences in abandonment linked to operational factors such as cutoff times or anchor days?

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline

# Data Loading & Structure

The dataset consists of eight CSV tables covering customer behavior, transactions, and supporting reference information. Three fact tables capture activity on MyCoke360: Google Analytics events (site visits, add/remove cart actions, purchases, and device/page details), Orders (materials ordered per customer, with order type and timestamps in both EST and UTC), and Sales (fulfilled transactions with pricing and profit measures). These are complemented by five dimension tables: Customer (account and channel attributes, sales office details), Cutoff Times (order cutoff policies by plant, office, and distribution mode), Material (product master data such as pack type, brand, flavor, and category), Operating Hours (current ordering frequency and anchor day/date by customer), and Visit Plan (historical anchor dates, frequencies, and sales office attributes). Collectively, these tables create a comprehensive view of both customer behavior and business processes adequetely enabling our analysis.

In [0]:
# Import fact tables
# google_analytics = spark.read.csv(
#     "/Volumes/workspace/default/capstone_data/google_analytics*.csv",
#     header=True,
#     inferSchema=False
# )
google_analytics = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/google_analytics.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
orders = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/orders.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
sales = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/sales.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)

# Import dimension tables
customers = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/customer.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
cutoff_times = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/cutoff_times.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
materials = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/material.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
operating_hours = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/operating_hours.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
visit_plan = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/visit_plan.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)

# Initial Quality Checks

## Data Types

In [0]:
display(google_analytics.limit(5))

In the google_analytics dataset, several columns require datatype adjustments before analysis. The EVENT_DATE column should be cast from its original yyyy-MM-dd string format to a proper date type. The EVENT_TIMESTAMP column is currently stored as a string in the yyyy-MM-ddTHH:mm:SSSZ format and will need to be converted into a timezone-aware timestamp. Additionally, the ITEMS column contains a json objects that will need to be parsed before they can be used.

In [0]:
items_schema = ArrayType(
    StructType([
        StructField("item_id", StringType()),
        StructField("quantity", StringType())
    ])
)

google_analytics = (
    google_analytics
    .withColumn("EVENT_DATE", to_date("EVENT_DATE", "yyyy-MM-dd"))
    .withColumn("EVENT_TIMESTAMP", to_timestamp("EVENT_TIMESTAMP", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
    .withColumn(
        "ITEMS",
        from_json(col("ITEMS"), items_schema).cast("array<struct<item_id:string, quantity:int>>")
    )
)
google_analytics.printSchema()

In [0]:
display(orders.limit(5))

orders = (
    orders
    .withColumn(
        "CREATED_DATE_EST",
        to_utc_timestamp(
            to_timestamp(col("CREATED_DATE_EST"), "yyyy-MM-dd"),
            "America/New_York"
        )
    )
    .withColumn("CREATED_DATE_UTC", to_timestamp("CREATED_DATE_UTC", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
    .withColumn("ORDER_QUANTITY", col("ORDER_QUANTITY").cast("double"))
)
display(orders.limit(5))
orders.printSchema()

Observations:
- CREATED_DATE_EST is a date in EST, original format yyyy-MM-dd
- CREATED_DATE_UTC is a timestamp in UTC, original format yyyy-MM-ddTHH:mm:SS.SSSZ
- ORDER_QUANTITY is a float, but likely clear to be set as an integer as it is a count

In [0]:
display(sales.limit(5))

sales = (
    sales
    .withColumn("POSTING_DATE", to_date("POSTING_DATE", "M/d/yyyy"))
    .withColumn("GROSS_PROFIT_DEAD_NET", col("GROSS_PROFIT_DEAD_NET").cast("double"))
    .withColumn("PHYSICAL_VOLUME", col("PHYSICAL_VOLUME").cast("double"))
    .withColumn("NSI_DEAD_NET", col("NSI_DEAD_NET").cast("double"))
)
display(sales.limit(5))
sales.printSchema()

Observations:
- POSTING_DATE is a date, likely utc as it has no other identifier (check notes), original format M/dd/yyyy
- GROSS_PROFIT_DEAD_NET is a monetary float value, consider comma removal before casting
- PHYSICAL_VOLUME is a float value, consider comma removal before casting
- NSI_DEAD_NET is a float value, consider comma removal before casting

In [0]:
display(customers.limit(5))

Where the customer table is concerned, the only numerical column is CUSTOMER_NUMBER. Since this is used as an identifier the standard is to leave it as a string datatype. The rest of the columns within the dataset are either categorical or descriptions which remain strings as well.

In [0]:
customers.printSchema()

In [0]:
display(cutoff_times.limit(5))

# cutoff_times = (
#     cutoff_times
#     .withColumn()
# )
cutoff_times.printSchema()

Observations:
- CUTOFFTIME__C is an AM/PM time, original format h:MM:SS AM
- SHIPPING_CONDITION_TIME looks to be a list of hours, may convert to integers (check first)

In [0]:
display(materials.limit(5))

# materials = (
#     materials
#     .withColumn()
# )
materials.printSchema()

Observations:
- No data types need changing here, they all should be strings

In [0]:
display(operating_hours.limit(5))

# operating_hours = (
#     operating_hours
#     .withColumn()
# )
operating_hours.printSchema()

Observations:
- FREQUENCY might be changed to integers (check first)
- DELIVERY_ANCHOR_DAY might be changed to integer week representations
- CALLING_ANCHOR_DATE is a date, original format d/MM/yyyy

In [0]:
display(visit_plan.limit(5))

# visit_plan = (
#     visit_plan
#     .withColumn()
# )
visit_plan.printSchema()

Observations:
- FREQUENCY might be changed to an integer (check unique values first)
- ELT_TS is a utc timestamp, original format yyyy-MM-ddTHH:mm:SS.SSSZ
- SNAPSHOT_DATE is a date, original format yyyy-MM-dd
- ANCHOR_DATE is a date, original format yyyy-MM-dd

## Missing Values

## Duplicates

# Data Cleaning

# Univariate Analysis

## Outlier Detection

# Bivariate Analysis

# Multivariate Analysis

# Target Variable Analysis

# Feature Engineering

# Statistical Testing

# Summary of Findings